# 03 — Feature Engineering

SQL-first aggregations (via DuckDB) across all 7 supplementary tables,
merged into the application base table with domain-driven derived features.

In [1]:
import sys
sys.path.insert(0, "../src")

import pathlib
import pandas as pd
import numpy as np
import duckdb

from feature_engineering import load_tables, run_sql_file, add_derived_features

RAW = pathlib.Path("../data/raw")
SQL = pathlib.Path("../sql")
OUT = pathlib.Path("../data/processed")

In [2]:
con = duckdb.connect()
load_tables(con, RAW)

  Loaded application_test (48,744 rows)
  Loaded application_train (307,511 rows)
  Loaded bureau (1,716,428 rows)
  Loaded bureau_balance (27,299,925 rows)
  Loaded credit_card_balance (3,840,312 rows)
  Loaded installments_payments (13,605,401 rows)
  Loaded POS_CASH_balance (10,001,358 rows)
  Loaded previous_application (1,670,214 rows)


## Run SQL Aggregations

In [3]:
agg_frames = {}
for sql_file in sorted(SQL.glob("*.sql")):
    name = sql_file.stem
    agg_frames[name] = run_sql_file(con, sql_file)
    print(f"{name}: {agg_frames[name].shape}")

bureau_agg: (305811, 15)
credit_card_agg: (92447, 16)
installment_features: (336935, 14)
pos_cash_agg: (334359, 14)
previous_app_agg: (338857, 16)


## Merge to Application Base

In [4]:
app = con.execute("SELECT * FROM application_train").fetchdf()
print(f"Base: {app.shape}")

for name, agg_df in agg_frames.items():
    if "SK_ID_CURR" in agg_df.columns:
        app = app.merge(agg_df, on="SK_ID_CURR", how="left")
        print(f"+ {name}: now {app.shape[1]} cols")

app = add_derived_features(app)
print(f"\nFinal shape: {app.shape}")

Base: (307511, 122)
+ bureau_agg: now 136 cols
+ credit_card_agg: now 151 cols
+ installment_features: now 164 cols
+ pos_cash_agg: now 177 cols
+ previous_app_agg: now 192 cols

Final shape: (307511, 199)


In [5]:
OUT.mkdir(parents=True, exist_ok=True)
app.to_csv(OUT / "train_features.csv", index=False)
print(f"✓ Saved train_features.csv ({app.shape[0]:,} rows × {app.shape[1]} cols)")
con.close()

✓ Saved train_features.csv (307,511 rows × 199 cols)
